# 🍁 LeafID: Colab runner

Runtime → Change runtime type → **T4 GPU**. Run top to bottom.

1. Setup  2. Data + fixed split  3. Experiments  4. Compare  5. Launch the app

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

REPO = "https://github.com/<your-username>/leafid.git"  # TODO: set after creating the GitHub repo
!git clone -q $REPO /content/leafid
%cd /content/leafid
!pip install -q -r requirements.txt && pip install -q -e .

## 2. Data
Downloads LeafSnap (~880 MB zip) with `kagglehub`. It's a public dataset, so no login is needed. If Kaggle ever asks for one, add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab **Secrets** (🔑 icon in the left bar).

`splits/splits.csv` is generated **once** and committed to the repo so every teammate trains and tests on the same images. Its paths are relative to the dataset root, so it works here and locally.

In [ ]:
import os
import kagglehub

# Optional credentials (only needed if anonymous download is ever refused)
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception:
    pass

path = kagglehub.dataset_download("xhlulu/leafsnap-dataset")
os.environ["LEAFID_DATA"] = path   # inherited by the !python commands below
print("dataset at", path)
!ls {path}

if not os.path.exists("splits/splits.csv"):
    !python -m leafid.splits
else:
    print("Using the committed splits/splits.csv")

## 3. Experiments
Each run saves `checkpoints/<run>.pt` and `reports/<run>/history.csv`. Comment out what you don't need.

In [ ]:
# Baselines trained from scratch
!python -m leafid.train --model simple_cnn --epochs 30 --lr 1e-3 --run simple_cnn
!python -m leafid.train --model res_cnn    --epochs 30 --lr 1e-3 --run res_cnn

In [ ]:
# Transfer learning with ImageNet-pretrained ResNet-50
!python -m leafid.train --model resnet50 --epochs 10 --freeze-epochs 10 --run resnet50_probe              # head only
!python -m leafid.train --model resnet50 --epochs 15 --freeze-epochs 3  --run resnet50_full               # two-stage fine-tune
!python -m leafid.train --model resnet50 --epochs 15 --freeze-epochs 3  --train-sources lab --run resnet50_labonly  # domain shift

## 4. Evaluate + compare

In [ ]:
import glob, json
import pandas as pd

for ckpt in sorted(glob.glob("checkpoints/*.pt")):
    !python -m leafid.evaluate --checkpoint {ckpt}

rows = []
for p in sorted(glob.glob("reports/*/metrics.json")):
    m = json.load(open(p))
    rows.append({"run": p.split("/")[-2], "model": m["model"],
                 "lab_top1": m["lab"]["top1"], "field_top1": m["field"]["top1"], "field_top5": m["field"]["top5"]})
pd.DataFrame(rows).round(3)

In [ ]:
# Optional: keep checkpoints after the Colab session ends
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/leafid && cp -r checkpoints reports /content/drive/MyDrive/leafid/

## 5. Launch the app
Opens a public `gradio.live` link. Add `--no-llm` for a faster start.

In [ ]:
!python app/app.py --checkpoint checkpoints/resnet50_full.pt --share